# Chapter 11 — Undoing & Recovery
> *You broke something. It happens. Git has a solution for almost everything.*
> *This chapter is the one you'll come back to at 11pm.* 🛟

---

> 💬 The Five Stages of Breaking Something with Git:
> 1. **Denial** — "It's fine, git status will show nothing"
> 2. **Anger** — "WHY IS THERE A CONFLICT I DIDN'T EVEN TOUCH THAT FILE"
> 3. **Bargaining** — "If I just delete the .git folder and re-clone..."
> 4. **Depression** — *stares at the diff in silence*
> 5. **Acceptance** — "Let me check git reflog"
>
> This chapter gets you to Step 5 in under a minute.

## 🗺️ Recovery Decision Tree — Start Here

```
Did you commit the thing you want to undo?
│
├── NO (changes are in working dir / staging)
│   ├── Unstage (still keep changes):   git restore --staged <file>
│   └── Discard changes completely:     git restore <file>   ← ⚠️ cannot undo this
│
└── YES (changes are in a commit)
    ├── It's the most recent commit?
    │   ├── Fix commit message:          git commit --amend -m "new message"
    │   ├── Undo commit → keep STAGED:   git reset --soft HEAD~1
    │   ├── Undo commit → keep in DIR:   git reset HEAD~1
    │   └── Undo + DELETE everything:    git reset --hard HEAD~1  ← ⚠️ dangerous
    │
    └── Already pushed to shared branch?
        └── git revert <hash>  ← ALWAYS use this for pushed commits. Always.
```

> 💬 Print this tree. Laminate it. Put it on your desk.
> It answers every "I made a mistake" scenario in Git.

## ⏪ `git reset` — The Time Machine

```bash
git reset --soft HEAD~1      # Undo last commit. Changes go back to STAGING. Nothing lost.
git reset HEAD~1             # Undo last commit. Changes go back to WORKING DIR. Nothing lost.
git reset --hard HEAD~1      # Undo last commit + DELETE ALL CHANGES. ⚠️ No recovery here.
git reset --hard HEAD~3      # Go back 3 commits (and delete everything since then)
git reset --hard <hash>      # Jump to a specific commit (delete everything after it)

# Find commit hashes:
git log --oneline
# a3f8c2d Fix login bug  ← this 7-char string is the hash
```

> 💬 `--soft` = time travel WITH your luggage (changes) still packed in the suitcase (staging).
> (no `--flag`) = time travel WITH your luggage unpacked in the hotel room (working dir).
> `--hard` = time travel and someone burned your luggage. Changes are gone. Hotel doesn't exist.
>
> Only use `--hard` when you WANT to lose the changes. Like starting over on a bad draft.
> Use `--soft` or default when you want to recommit with a better approach.

## ↩️ `git revert` — The Safe Undo for Shared Commits

> `git reset` rewrites history → **NEVER use on commits already pushed to shared branches**
> `git revert` creates a NEW commit that undoes → **safe for absolutely any situation** ✅

```bash
git revert <hash>           # Creates a new commit that undoes that specific commit
git revert HEAD             # Undo the very most recent commit (safely)
git revert HEAD --no-edit   # Revert without opening the message editor
```

> 💬 Why not just `git reset --hard` on a pushed commit?
>
> Your teammates already pulled those commits into their repos.
> If you delete them from your history and force-push,
> their repos now have commits that only exist on their machines.
> When they push, the histories diverge. Everyone has a bad day.
>
> `git revert` keeps the original commit in history.
> It just adds a new commit that says "undo that."
> Clean. Transparent. Safe. Professional.

## 🚨 `git reflog` — The Ultimate Safety Net

Even after `git reset --hard`, your old commits survive in reflog for approximately 90 days.
Reflog = a log of every time HEAD moved. Every commit, every reset, every checkout.
It remembers things even when the branch doesn't.

```bash
git reflog
# output:
# a3f8c2d HEAD@{0}: reset: moving to HEAD~1
# b2e1a9f HEAD@{1}: commit: Add login feature   ← your "lost" commit!
# c5d9f30 HEAD@{2}: commit: Initial commit

# Recover the "lost" commit:
git reset --hard b2e1a9f      # Jump directly back to it
# OR create a new branch from it (safer — doesn't move HEAD):
git branch rescue-branch b2e1a9f
```

> 💬 `git reflog` is the reason senior developers don't panic when someone says
> "I accidentally ran `git reset --hard` and now 2 days of work are gone."
>
> They open reflog. They find the hash. They recover everything.
> The word "lost" in Git usually means "temporarily misplaced."
>
> The only TRULY lost commits are those that never got committed in the first place.
> Which is why we commit early and commit often.

## 📊 Quick Comparison Table

| Command | What happens | Safe after push? |
|---------|-------------|-----------------|
| `git restore <file>` | Discard working dir changes | ✅ Only local |
| `git restore --staged` | Unstage (keep changes) | ✅ Only local |
| `git reset --soft` | Undo commit → into staging | ⚠️ LOCAL only |
| `git reset --hard` | Undo commit + delete | ❌ Never on pushed |
| `git revert` | New undo commit | ✅ Always safe |